# 07 — Compiled replay and larger-data pressure test
Compare the optimized replay with notebook 06 before extending to **50,000 real AAPL trades**. Preserve score formulas, every-event discovery, immutable band bounds, strict breaks and opposite-side retests. Keep history and opacity, but avoid a score dictionary for every active band at every event.

This notebook measures implementation speed and long-history behavior; it does not validate trading profitability. It preserves notebook 06. Compilation, source loading, reference replay, optimized replay, audits and rendering are timed separately. All outputs belong outside the repository.


In [ ]:
import os,sys,json,time,hashlib,gc,cProfile,pstats,io
os.environ["PYTHONDONTWRITEBYTECODE"]="1"
sys.dont_write_bytecode=True
from pathlib import Path
from datetime import datetime,timezone
from zoneinfo import ZoneInfo
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import psutil
from IPython.display import display
REPO=Path("D:/TradingCodes/quant-research-workbench")
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from research.level_book.v8.pressure_source import load_pressure_sample
from research.level_book.v8.pressure_engine import run_fast,load_reference,compare_reference,audit_replay,save_compact
ROOT=Path("D:/TradingML/runtimes")
assert ROOT.is_dir(),"Required runtime root unavailable"
RUN=ROOT/"level-book-v8/pressure-test"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True,exist_ok=False)
SOURCE_RUN=ROOT/"level-book-v8/pressure-test/source-50000"
ORIGINAL_RUN=ROOT/"level-book-v8/reaction-demo/20261020T204419563339Z"
COUNT=50000
TIERS=[2000,10000,25000,50000]
DISCOVERY=200
print("Run directory:",RUN)
pl.Config.set_tbl_rows(15);pl.Config.set_tbl_cols(16)


## 1. Load more data from the same canonical authority
Reuse the saved 50,000-row certified snapshot if present; otherwise read the first 50,000 eligible AAPL trades of 2026-08-20 from `market_sip_compact.events_2026`. The separate larger-sample loader keeps notebook 01's eligibility/provenance checks and bounded read-only limits. The first hour had only 45,039 eligible trades, so the permitted read window is 09:30–16:00 New York time. No raw-file fallback or database write occurs.

Verify the snapshot hash, ordering, unique ordinal, and exact agreement with the original 2,000 trades. This prevents a speed comparison on different inputs. Source-window audit counts include rejected trades; the requested eligible prefix is intentionally bounded and is not a full-session dataset.


In [ ]:
load_start=time.perf_counter()
if not (SOURCE_RUN/"source_trades.parquet").is_file():
    rows,provenance=load_pressure_sample(count=COUNT)
    SOURCE_RUN.mkdir(parents=True,exist_ok=True)
    source=pl.DataFrame(rows).with_row_index("event",offset=1).with_columns(
        pl.from_epoch("time_us",time_unit="us").dt.replace_time_zone("UTC").alias("time"),
        (pl.col("price_units")/10000).alias("price"))
    source.write_parquet(SOURCE_RUN/"source_trades.parquet")
    (SOURCE_RUN/"source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
else:
    source=pl.read_parquet(SOURCE_RUN/"source_trades.parquet")
    provenance=json.loads((SOURCE_RUN/"source_provenance.json").read_text(encoding="utf-8"))
records=source.select("ordinal","time_us","price_units","volume").to_dicts()
for r in records:
    if float(r["volume"]).is_integer():r["volume"]=int(r["volume"])
source_hash=hashlib.sha256(json.dumps(records,sort_keys=True).encode()).hexdigest()
assert source_hash==provenance["sample_sha256"]
assert source.height==COUNT and provenance["ticker"]=="AAPL" and provenance["day"]=="2026-08-20"
original=pl.read_parquet(ORIGINAL_RUN/"source_trades.parquet")
for column in ["ordinal","time_us","price_units","volume"]:
    assert np.array_equal(source[column].head(2000).to_numpy(),original[column].to_numpy())
assert source["ordinal"].n_unique()==source.height
assert source.select("time_us","ordinal").equals(source.sort("time_us","ordinal").select("time_us","ordinal"))
session_open=datetime.fromisoformat(provenance["day"]+"T09:30:00").replace(tzinfo=ZoneInfo("America/New_York"))
SESSION_OPEN_US=int(session_open.timestamp()*1e6)
load_seconds=time.perf_counter()-load_start
print("Verified events:",source.height,"unique prices:",source["price_units"].n_unique(),"load/verification seconds:",load_seconds)
print("Observed time range:",source["time"][0],"to",source["time"][-1])
display(source.head(10));display(pl.DataFrame(provenance["audit"]))
(RUN/"source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
del records;gc.collect()


## 2. Vectorized preparation and the actual bottlenecks
Below are the six agreed differences. A forward reaction on row i becomes available only when row i+1 arrives; the compiled updater obeys that ordering. Integer price coordinates avoid equality errors. Coordinate compression may allocate slots for prices from the loaded array, but an observed-price mask excludes unseen slots from all profiles and decisions.

Optimization changes implementation, not the discovery cadence:

| Notebook 06 work | Notebook 07 replacement |
|---|---|
| Python plateau dictionaries and repeated scans | Compiled numeric plateau arrays with linear neighboring-valley passes |
| Repeated sum over each candidate range | Prefix sums and constant-time range totals |
| Repeated checks against every existing band | Permanent price-to-band owner index |
| Rebuild profiles when nothing changed | Rebuild only after new observed price or completed reversal evidence |
| Python loops over every transition | Compiled chronological state loop; keep causality |
| One score dictionary per band per event | Evidence changes per band/role plus shared normalization series; derive age analytically |

Price topology and numeric values still need scanning when evidence changes. This is not an O(1) discovery algorithm. Exact transition order is tested; floating range sums use a tight numerical tolerance. JIT compilation has a separate first-run cost and writes no source-tree cache.


In [ ]:
step1=source.with_columns(
    ((pl.col("time_us")-pl.col("time_us").shift(1))/1e6).alias("time_backward"),
    ((pl.col("time_us").shift(-1)-pl.col("time_us"))/1e6).alias("time_forward"),
    ((pl.col("price_units")-pl.col("price_units").shift(1))/10000).alias("price_backward"),
    ((pl.col("price_units").shift(-1)-pl.col("price_units"))/10000).alias("price_forward"),
    (pl.col("volume")-pl.col("volume").shift(1)).alias("volume_backward"),
    (pl.col("volume").shift(-1)-pl.col("volume")).alias("volume_forward"),
).with_columns(pl.col("price_backward").sign().alias("arrival"),pl.col("price_forward").sign().alias("departure"))
display(step1.select("event","time_backward","time_forward","price_backward","price_forward","volume_backward","volume_forward","arrival","departure").head(12))
assert step1["departure"][-1] is None
reference_fn=load_reference(REPO/"research/level_book/v8/06_unbroken_age_strength.ipynb",SESSION_OPEN_US)
profiler=cProfile.Profile();profiler.enable()
profiled=reference_fn(source.head(2000));profiler.disable()
report=io.StringIO();pstats.Stats(profiler,stream=report).sort_stats("cumtime").print_stats(12)
print(report.getvalue());(RUN/"baseline_profile.txt").write_text(report.getvalue(),encoding="utf-8")
del profiled;gc.collect()


## 3. Reference parity before pressure testing
The 200-event warm-up measures compilation separately. Then compare 2,000 and 10,000 real events with notebook 06: exact role periods, bounds, origins, starts, breaks, parents and deferred-candidate counts, plus every saved score/evidence sample. Stable IDs use a one-to-one mapping because notebook 07 assigns compact consecutive band IDs; period IDs and transition order must match exactly.

Baseline timings below are unprofiled wall times. A process RSS reading includes Python, compilation and data, not just the engine. Compact-array bytes are reported separately; they are not a peak-RSS claim.


In [ ]:
start=time.perf_counter();warmup=run_fast(source.head(200),SESSION_OPEN_US)
compile_seconds=time.perf_counter()-start
print("Compilation plus first 200-event call:",compile_seconds,"seconds",flush=True)
parity=[];baseline_times={}
for n in [2000,10000]:
    print("Reference/parity events:",n,flush=True)
    start=time.perf_counter();baseline=reference_fn(source.head(n));baseline_times[n]=time.perf_counter()-start
    fast=run_fast(source.head(n),SESSION_OPEN_US)
    checks=compare_reference(fast,baseline)
    parity.append(dict(events=n,reference_seconds=baseline_times[n],**checks))
    print(parity[-1],flush=True)
    del baseline,fast;gc.collect()
display(pl.DataFrame(parity))
(RUN/"parity.json").write_text(json.dumps(parity,indent=2),encoding="utf-8")


## 4. Pressure-test increasing real-event prefixes
Run the same optimized method at increasing sizes. Save one receipt per completed tier so a later failure does not erase completed measurements. The source cache survives restarts; analysis runs are independent and can be rerun. Saved compact results are analysis caches, **not** streaming-resume checkpoints.

The next section audits the largest run by directly scanning actual trades for every period's first break and validating retest entry, rather than trusting the compiled state machine alone.


In [ ]:
measurements=[]
for n in TIERS:
    print("Pressure tier:",n,flush=True)
    start=time.perf_counter();result=run_fast(source.head(n),SESSION_OPEN_US);wall=time.perf_counter()-start
    zones,active,pending,periods=map(int,result.counts[-1])
    row=dict(events=n,unique_prices=source.head(n)["price_units"].n_unique(),wall_seconds=wall,
        kernel_seconds=result.seconds,events_per_second=n/wall,compact_array_mb=result.array_bytes/1e6,
        process_rss_mb=psutil.Process().memory_info().rss/1e6,zones=zones,active=active,pending=pending,
        role_periods=periods,profile_rebuilds=int(result.profile_builds),merges=0,
        deferred_overlap=int(result.deferred[0]),deferred_wrong_side=int(result.deferred[1]),
        reference_seconds=baseline_times.get(n),speedup=baseline_times[n]/wall if n in baseline_times else None)
    measurements.append(row)
    (RUN/f"tier_{n}.json").write_text(json.dumps(row,indent=2),encoding="utf-8")
    print(row,flush=True)
display(pl.DataFrame(measurements))
benchmark=pl.DataFrame(measurements);benchmark.write_parquet(RUN/"benchmarks.parquet")
save_compact(result,RUN/"compact_result.npz")


## 5. What happens to older levels? No automatic merging
Band boundaries and identity remain stable. A band can change role after a break/retest, but nearby historical bands are not merged. A pending band reserves its price region indefinitely; overlapping candidates are counted as deferred. This is the current notebook-06 rule, not a claim that indefinite retention is desirable.

Inspect active versus pending counts, zone growth, pending age and overlap deferrals. Deferrals count repeated candidate observations, not unique rejected levels. Large pending populations indicate a future lifecycle/retirement decision, not a performance bug we should silently hide by merging bands.


In [ ]:
audit_start=time.perf_counter();audit=audit_replay(result);audit_seconds=time.perf_counter()-audit_start
# A separately replayed 10,000-event prefix must equal the full run's as-of view.
short=run_fast(source.head(10000),SESSION_OPEN_US)
for a,b in zip(short.state_at(10000),result.state_at(10000)):
    assert a.equals(b)
# Changing only future prices cannot affect earlier coordinates, decisions or scores.
perturbed=source.head(2500).with_columns(pl.when(pl.col("event")>2000).then(pl.col("price_units")+100000).otherwise(pl.col("price_units")).alias("price_units"))
altered=run_fast(perturbed,SESSION_OPEN_US);original_prefix=run_fast(source.head(2000),SESSION_OPEN_US)
for a,b in zip(altered.state_at(2000),original_prefix.state_at(2000)):assert a.equals(b)
final_state,final_waiting=result.state_at(COUNT)
if final_waiting.height:
    final_waiting=final_waiting.with_columns(pl.Series("waiting_seconds",(result.times[-1]-result.times[final_waiting["since_event"].to_numpy()-1])/1e6))
print("Independent audit:",audit,"seconds:",audit_seconds)
display(final_state.group_by("role","status").len())
display(final_waiting.sort("waiting_seconds",descending=True).head(15))
# Show one old band's complete role lineage instead of treating role periods as new bands.
lineage_id=int(np.bincount(result.periods[:,0]).argmax())
display(final_state.filter(pl.col("band_id")==lineage_id).select("band_id","period_id","role","start_event","broken_event","status","strength"))
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].plot(np.arange(1,COUNT+1),result.counts[:,0],label="All stable bands")
axes[0].plot(np.arange(1,COUNT+1),result.counts[:,1],label="Active")
axes[0].plot(np.arange(1,COUNT+1),result.counts[:,2],label="Waiting")
axes[0].set(xlabel="Event",ylabel="Bands",title="Band accumulation; zero merges");axes[0].legend()
axes[1].plot(benchmark["events"],benchmark["wall_seconds"],marker="o")
axes[1].set(xlabel="Events",ylabel="Seconds",title="Warm replay wall time")
fig.savefig(RUN/"pressure_summary.png",dpi=140,bbox_inches="tight");plt.show();plt.close(fig)

# Edge streams exercise absent peaks and repeated/tied-price topology.
edge_checks=[]
for name,prices in [("flat",np.full(500,1000000)),("alternating",np.resize([1000000,1000100,1000200,1000100],500)),("rising",np.arange(1000000,1000500))]:
    edge=source.head(500).with_columns(pl.Series("price_units",prices))
    edge_fast=run_fast(edge,SESSION_OPEN_US)
    edge_checks.append(dict(case=name,**compare_reference(edge_fast,reference_fn(edge))))
    edge_fast.state_at(500)  # Empty results must retain the normal table schema.
print("Edge checks:",edge_checks)


## 6. Interactive final levels and historical replay
One support/resistance chart retains the as-of slider, independent thresholds and time-varying opacity. The default view shows **active periods** at the end of the loaded data. Choose all historical periods to inspect breaks and role changes; optionally show pending bands. No legend is placed on the price chart.

Scores and tables are exact as of the selected event. For large views, opacity is evaluated on a bounded display grid (target 60,000 intervals across visible periods), retaining each period's start/end; its stride is printed. This is a rendering approximation only, not reduced discovery or scoring cadence. Short sample views normally use every event. Thresholds make weak bands dotted and control margin labels; they do not delete the state. At most 20 strongest qualifying bands receive labels to keep the chart readable; all bands remain drawn and all values are available in the table/runtime output.


In [ ]:
import ipywidgets as widgets
from matplotlib.collections import LineCollection,PolyCollection
from matplotlib.colors import to_rgba
colors={"support":"#16803c","resistance":"#c53030"};chart_state={}
def draw_pressure(events,support,resistance,view,show_waiting):
    render_start=time.perf_counter();state,waiting=result.state_at(events)
    thresholds={"support":support,"resistance":resistance}
    state=state.join(pl.DataFrame({"role":list(thresholds),"threshold":list(thresholds.values())}),on="role").with_columns(
        pl.when(pl.col("status")=="broken").then(pl.lit("broken"))
          .when(pl.col("strength")<pl.col("threshold")).then(pl.lit("active below threshold"))
          .otherwise(pl.lit("active above threshold")).alias("display_status"))
    drawn=state if view=="All historical periods" else state.filter(pl.col("status")=="active")
    duration=int((drawn["draw_end_event"]-drawn["start_event"]).sum())
    stride=max(1,int(np.ceil(duration/60000)))
    fig,ax=plt.subplots(figsize=(18,10));fig.subplots_adjust(right=.80)
    ax.plot(source["event"].head(events),source["price"].head(events),color="black",lw=.6)
    edges=[];edgecolors=[];styles=[];polygons=[];facecolors=[];break_x=[];break_y=[];break_colors=[]
    for row in drawn.iter_rows(named=True):
        start=row["start_event"];end=row["draw_end_event"];pid=row["period_id"]
        lo=row["lower_units"]/10000;hi=row["upper_units"]/10000;role=row["role"]
        grid=np.unique(np.r_[np.arange(start,end,stride,dtype=np.int64),end])
        score=result.samples(pid,grid)[0]
        for j in range(len(grid)-1):
            a=int(grid[j]);b=int(grid[j+1]);s=float(score[j])
            edges.extend([[(a,lo),(b,lo)],[(a,hi),(b,hi)]])
            edgecolors.extend([to_rgba(colors[role],.08+.82*s)]*2)
            styles.extend([":" if s<thresholds[role] else "--" if role=="resistance" else "-"]*2)
            if hi>lo:
                polygons.append([(a,lo),(b,lo),(b,hi),(a,hi)]);facecolors.append(to_rgba(colors[role],.02+.3*s))
        if row["status"]=="broken":
            break_x.append(end);break_y.append((lo+hi)/2);break_colors.append(colors[role])
    if polygons:ax.add_collection(PolyCollection(polygons,facecolors=facecolors,edgecolors="none"))
    if edges:ax.add_collection(LineCollection(edges,colors=edgecolors,linewidths=.8,linestyles=styles))
    if break_x:ax.scatter(break_x,break_y,marker="x",s=8,c=break_colors)
    if show_waiting and waiting.height:
        waiting_segments=[[(row["since_event"],(row["lower_units"]+row["upper_units"])/20000),
                           (events,(row["lower_units"]+row["upper_units"])/20000)] for row in waiting.iter_rows(named=True)]
        ax.add_collection(LineCollection(waiting_segments,colors="gray",linewidths=.5,alpha=.3,linestyles=":"))
    ax.autoscale_view()
    qualified=state.filter(pl.col("display_status")=="active above threshold")
    labels=qualified.sort("strength",descending=True).head(20).with_columns(((pl.col("lower_units")+pl.col("upper_units"))/20000).alias("center")).sort("center")
    low,high=ax.get_ylim();ys=np.linspace(low+.03*(high-low),high-.03*(high-low),labels.height) if labels.height>1 else labels["center"].to_numpy()
    for row,y in zip(labels.iter_rows(named=True),ys):
        ax.annotate(f"{row['role']} B{row['band_id']} | {row['strength']:.3f}",xy=(events,row["center"]),xytext=(1.01,y),textcoords=ax.get_yaxis_transform(),color=colors[row["role"]],fontsize=8,annotation_clip=False,
                    arrowprops=dict(arrowstyle="-",color=colors[row["role"]],lw=.4,relpos=(0,.5)))
    ax.set(xlim=(1,events),xlabel="As-of event",ylabel="Price ($)",title="Notebook 07: stable bands, updated reactions, opacity = strength")
    ax.grid(alpha=.12)
    fig.savefig(RUN/"pressure_levels.png",dpi=120,bbox_inches="tight");plt.show();plt.close(fig)
    elapsed=time.perf_counter()-render_start
    print(f"As of {events:,} events | opacity stride {stride} events | labeled {labels.height}/{qualified.height} qualified bands | render {elapsed:.2f}s")
    display(state.group_by("role","display_status").len().sort("role","display_status"))
    display(waiting.group_by("waiting_for").len())
    display(state.filter(pl.col("status")=="active").with_columns(
        (pl.col("lower_units")/10000).alias("lower"),(pl.col("upper_units")/10000).alias("upper"))
        .select("band_id","role","lower","upper","trade_count","base_score","strength","display_status").sort("strength",descending=True))
    state.write_parquet(RUN/"current_state.parquet");waiting.write_parquet(RUN/"current_waiting.parquet")
    chart_state.update(event=events,state=state,waiting=waiting,stride=stride,render_seconds=elapsed,renders=chart_state.get("renders",0)+1)
    (RUN/"view_settings.json").write_text(json.dumps(dict(events=events,support=support,resistance=resistance,view=view,show_waiting=show_waiting,stride=stride,render_seconds=elapsed),indent=2),encoding="utf-8")
controls={"events":widgets.IntSlider(value=COUNT,min=DISCOVERY,max=COUNT,step=1,description="As-of event",continuous_update=False,layout=widgets.Layout(width="900px")),
          "support":widgets.FloatSlider(value=.5,min=0,max=1,step=.01,description="Support",continuous_update=False),
          "resistance":widgets.FloatSlider(value=.5,min=0,max=1,step=.01,description="Resistance",continuous_update=False),
          "view":widgets.Dropdown(options=["Active periods","All historical periods"],value="Active periods",description="Show"),
          "show_waiting":widgets.Checkbox(value=False,description="Show waiting bands")}
output=widgets.interactive_output(draw_pressure,controls)
display(widgets.VBox([controls["events"],widgets.HBox([controls["support"],controls["resistance"]]),widgets.HBox([controls["view"],controls["show_waiting"]]),output]))


## 7. Record evidence and limits
The parity and independent audits establish implementation behavior on these samples. They do not establish general market usefulness, constant-time scaling, or production checkpoint readiness. The retained-zone policy still has no merge or expiry. Larger data, different tickers and trading days can expose different topology and growth patterns.


In [ ]:
manifest=dict(method="07-compiled-evolving-bands",source_sha256=source_hash,events=COUNT,
    compile_seconds=compile_seconds,load_seconds=load_seconds,parity=parity,benchmarks=measurements,
    audit=audit,edge_checks=edge_checks,audit_seconds=audit_seconds,session_open=session_open.isoformat(),merges=0,
    notebook_sha256=hashlib.sha256((REPO/"research/level_book/v8/07_compiled_replay_pressure_test.ipynb").read_bytes()).hexdigest(),
    engine_sha256=hashlib.sha256((REPO/"research/level_book/v8/pressure_engine.py").read_bytes()).hexdigest(),
    baseline_sha256=hashlib.sha256((REPO/"research/level_book/v8/06_unbroken_age_strength.ipynb").read_bytes()).hexdigest(),
    git_commit=__import__("subprocess").check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip(),checks="passed")
(RUN/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("PASS: source identity, exact transitions, score parity, direct break/retest audits, prefix isolation")
print("Evidence:",RUN)
